In [49]:
import pandas as pd
import numpy as np

Loading the dataset

In [50]:
#loading csv file
df = pd.read_csv("dirty_cafe_sales.csv")

In [51]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [52]:
# BEFORE CLEANING Statistics Summary
before_stats = {
    'null_count': df.isnull().sum().sum(),
    'duplicate_count': df.duplicated().sum(),
    'row_count': df.shape[0],
    'dtypes': df.dtypes
}
print("Before:", before_stats)

Before: {'null_count': 6826, 'duplicate_count': 0, 'row_count': 10000, 'dtypes': Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object}


Data Quality Check

In [53]:
#count of nulls per column
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Nulls Handling

In [54]:
group1 = ['Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

In [55]:
#setting all "Unknown" and "Error" Value to Null
df[group1] = df[group1].replace(['UNKNOWN', 'ERROR'], pd.NA)

In [56]:
df[group1].isnull().sum()

Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [57]:
#check for distinct data in the Item column
df['Item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', <NA>, 'Sandwich',
       nan, 'Juice', 'Tea'], dtype=object)

In [58]:
#relation between Item, Quantity, Price Per Unit, Total Spent using mathematical calculation
#Total Spent = Quantity * Price Per Unit
group2 = ['Quantity', 'Price Per Unit', 'Total Spent']

In [59]:
#check for data type for the columns
df.dtypes

Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

In [60]:
#converting the group2 columns to numeric data type
df[group2] = df[group2].apply(pd.to_numeric, errors= 'coerce')

In [61]:
#Using the for loop to handle missing data in group2 column using calculation where only one value is unknown..
for i in df.index:
    a = df.loc[i, 'Quantity']
    b = df.loc[i, 'Price Per Unit']
    c = df.loc[i, 'Total Spent']

    if pd.isna(a) and pd.notna(b) and pd.notna(c) and b != 0:
        df.loc[i, 'Quantity'] = c / b

    elif pd.isna(b) and pd.notna(a) and pd.notna(c) and a != 0:
        df.loc[i, 'Price Per Unit'] = c / a

    elif pd.isna(c) and pd.notna(a) and pd.notna(b):
        df.loc[i, 'Total Spent'] = a * b


In [62]:
#recheck to comfirm null values 
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity              38
Price Per Unit        38
Total Spent           40
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [63]:
#cross-checking to comfirm only rows in group2 with two unknown are nulls..
df[df[group2].isnull().any(axis = 1)]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-store,2023-10-20
236,TXN_8562645,Salad,NaN,5.0,NaN,NaN,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
629,TXN_9289174,Cake,NaN,NaN,12.0,Digital Wallet,In-store,2023-12-30
641,TXN_2962976,Juice,NaN,3.0,NaN,NaN,NaN,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,NaN,Takeaway,2023-05-14
912,TXN_1575608,Sandwich,NaN,NaN,20.0,<NA>,Takeaway,2023-01-05
1008,TXN_7225428,Tea,NaN,NaN,3.0,Credit Card,Takeaway,2023-03-07
1436,TXN_7590801,Tea,NaN,NaN,6.0,Cash,Takeaway,<NA>
1482,TXN_3593060,Smoothie,NaN,NaN,16.0,Cash,NaN,2023-03-05


In [64]:
#checking for average price for each item
df.groupby('Item')['Price Per Unit'].mean()

Item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: Price Per Unit, dtype: float64

In [65]:
df.isnull().sum()

Transaction ID         0
Item                 969
Quantity              38
Price Per Unit        38
Total Spent           40
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [66]:
#Using the relationship between Item and price to fill missing price data using mode fiilna method for null items 

item_to_price = df.groupby('Item')['Price Per Unit'].agg(lambda x: x.mode().iloc[0])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(item_to_price))

In [67]:
#check for null values 'Price Per Unit' column
df['Price Per Unit'].isnull().sum()

6

In [68]:
#Using relationship between Item and price to fill missing Item data using mode fillna method for null items
price_to_item = df.groupby('Price Per Unit')['Item'].agg(lambda x: x.mode().iloc[0])
df['Item'] = df['Item'].fillna(df['Price Per Unit'].map(price_to_item))

In [69]:
#check for null values 'Item' column
df['Item'].isnull().sum()

6

In [70]:
#using for loop to calculate missing data for quantity and Total spent
for i in df.index:
    p = df.loc[i, 'Quantity']
    q = df.loc[i, 'Price Per Unit']
    r = df.loc[i, 'Total Spent']

    if pd.isna(p) and pd.notna(q) and pd.notna(r):
        df.loc[i, 'Quantity'] = r / q

    if pd.isna(q) and pd.notna(p) and pd.notna(r):
        df.loc[i, 'Price Per Unit'] = r / p

    if pd.isna(r) and pd.notna(p) and pd.notna(q):
        df.loc[i, 'Total Spent'] = p * q
    


In [71]:
#check for null values for Quantity and Price Per Unit
df.isnull().sum()

Transaction ID         0
Item                   6
Quantity              23
Price Per Unit         6
Total Spent           23
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [72]:
df[df['Item'].isna()]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
1761,TXN_3611851,NaN,4.0,NaN,NaN,Credit Card,NaN,2023-02-09
2289,TXN_7524977,NaN,4.0,NaN,NaN,<NA>,NaN,2023-12-09
3779,TXN_7376255,NaN,NaN,NaN,25.0,NaN,In-store,2023-05-27
4152,TXN_9646000,NaN,2.0,NaN,NaN,NaN,In-store,2023-12-14
7597,TXN_1082717,NaN,NaN,NaN,9.0,Digital Wallet,In-store,2023-12-13
9819,TXN_1208561,NaN,NaN,NaN,20.0,Credit Card,NaN,2023-08-19


In [73]:
df.groupby('Item')['Transaction ID'].count().sort_values(ascending = False)

Item
Juice       1418
Sandwich    1358
Coffee      1291
Salad       1272
Cookie      1213
Tea         1207
Cake        1139
Smoothie    1096
Name: Transaction ID, dtype: int64

In [74]:
group3 = ['Item','Quantity', 'Price Per Unit', 'Total Spent']

In [75]:
#using the dropna function 
df = df.dropna(subset = group3)

Rows with missing Item, Price, Quantity, or Total were dropped 
because they represent incomplete transactions and make up less than 1% of the dataset. 
This helps avoid calculation errors while preserving 99% of the data.

In [76]:
#final null check for the first five rows
df.isnull().sum()

Transaction ID         0
Item                   0
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3168
Location            3952
Transaction Date     460
dtype: int64

Null Handling for the 'Payment Method', 'Location' and 'Transaction Date' columns respectively

In [77]:
df['Payment Method'].unique()

array(['Credit Card', 'Cash', <NA>, 'Digital Wallet', nan], dtype=object)

In [78]:
df['Location'].unique()

array(['Takeaway', 'In-store', <NA>, nan], dtype=object)

In [79]:
#using the Fillna funtion with 'Unknown' for both Payment method and Location
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')



'Unknown was used to fill both columns because Missing values in categorical columns like Payment Method and Location cannot be filled with a mean or median like numeric data. Using Unknown preserves the row without making false assumptions about the actual payment method or location. 

In [80]:
df = df.dropna(subset = ['Transaction Date']).copy()

Transaction Date is a critical field, this is because without a date, a transaction record is invalid and cannot be used for analysis, sorting, or trend reporting. Date cannot guessed or imputed therefore the rows without date is dropped. 

In [81]:
#null check for all column
df.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Duplicate Values Check

In [82]:
df.duplicated().sum()

0

In [83]:
#converting 'transaction date' data type to date
df.loc[:, 'Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors = 'coerce')

In [84]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08 00:00:00
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16 00:00:00
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19 00:00:00
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27 00:00:00
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11 00:00:00
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Unknown,Unknown,2023-08-30 00:00:00
9996,TXN_9659401,Cookie,3.0,1.0,3.0,Digital Wallet,Unknown,2023-06-02 00:00:00
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Unknown,2023-03-02 00:00:00
9998,TXN_7695629,Cookie,3.0,1.0,3.0,Digital Wallet,Unknown,2023-12-02 00:00:00


Standardization

In [85]:
#data type conversion
df['Transaction ID']= df['Transaction ID'].astype(str)
df['Item']= df['Item'].astype(str)
df['Payment Method']= df['Payment Method'].astype(str)
df['Location']= df['Location'].astype(str)

In [86]:
#conversion to lowecase for consistency and uniformity
df['Transaction ID']= df['Transaction ID'].str.lower()
df['Item']= df['Item'].str.lower()
df['Payment Method']= df['Payment Method'].str.lower()
df['Location']= df['Location'].str.lower()

In [87]:
df.head(2)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,txn_1961373,coffee,2.0,2.0,4.0,credit card,takeaway,2023-09-08 00:00:00
1,txn_4977031,cake,4.0,3.0,12.0,cash,in-store,2023-05-16 00:00:00


In [88]:
#stripping white spaces
df['Transaction ID']= df['Transaction ID'].str.strip()
df['Item']= df['Item'].str.strip()
df['Payment Method']= df['Payment Method'].str.strip()
df['Location']= df['Location'].str.strip()

In [89]:
df.head(2)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,txn_1961373,coffee,2.0,2.0,4.0,credit card,takeaway,2023-09-08 00:00:00
1,txn_4977031,cake,4.0,3.0,12.0,cash,in-store,2023-05-16 00:00:00


Outlier detection

In [90]:
#Detect Outliers Using IQR

for check in group2:
    Q1 = df[check].quantile(0.25)
    Q3 = df[check].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[check] < lower) | (df[check] > upper)]
    print(f"{check}: {len(outliers)} outliers | Bounds: [{lower:.2f}, {upper:.2f}]")

Quantity: 0 outliers | Bounds: [-1.00, 7.00]
Price Per Unit: 0 outliers | Bounds: [-1.00, 7.00]
Total Spent: 259 outliers | Bounds: [-8.00, 24.00]


- For 'Quantity', it shows 0 outliers meaning all values are within the normal range of -1 to 7. No bulk purchase anomalies.
- For 'Price Per Unit', it shows 0 outliers meaning all prices are consistent, no pricing errors.
- For 'Total Spent', it shows 259 outliers meaning 259 transactions are above 24. This is expected because Total Spent = Quantity * Price, so even small increases in both push the total high.

Therefore my decision is to retain all. 


In [91]:
#After Cleaning Statistics Summary
After_stats = {
    'null_count': df.isnull().sum().sum(),
    'duplicate_count': df.duplicated().sum(),
    'row_count': df.shape[0],
    'dtypes': df.dtypes
}
print("After:", After_stats)

After: {'null_count': 0, 'duplicate_count': 0, 'row_count': 9514, 'dtypes': Transaction ID       object
Item                 object
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
dtype: object}


In [92]:
# AFTER CLEANING - Compare
import pandas as pd

summary_df = pd.DataFrame({
    'Metric': ['Null Count', 'Duplicate Count', 'Row Count', 'Dtype Example (Transaction Date)'],
    'Before Cleaning': [
        before_stats['null_count'],
        before_stats['duplicate_count'],
        before_stats['row_count']
       
    ],
    'After Cleaning': [
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df.shape[0]
       
    ]
})

summary_df

ValueError: All arrays must be of the same length

In [93]:
df.to_csv('cleaned_dataset.csv', index= False)

In [94]:
check = pd.read_csv('cleaned_dataset.csv')

In [95]:
check

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,txn_1961373,coffee,2.0,2.0,4.0,credit card,takeaway,2023-09-08 00:00:00
1,txn_4977031,cake,4.0,3.0,12.0,cash,in-store,2023-05-16 00:00:00
2,txn_4271903,cookie,4.0,1.0,4.0,credit card,in-store,2023-07-19 00:00:00
3,txn_7034554,salad,2.0,5.0,10.0,unknown,unknown,2023-04-27 00:00:00
4,txn_3160411,coffee,2.0,2.0,4.0,digital wallet,in-store,2023-06-11 00:00:00
...,...,...,...,...,...,...,...,...
9509,txn_7672686,coffee,2.0,2.0,4.0,unknown,unknown,2023-08-30 00:00:00
9510,txn_9659401,cookie,3.0,1.0,3.0,digital wallet,unknown,2023-06-02 00:00:00
9511,txn_5255387,coffee,4.0,2.0,8.0,digital wallet,unknown,2023-03-02 00:00:00
9512,txn_7695629,cookie,3.0,1.0,3.0,digital wallet,unknown,2023-12-02 00:00:00


In [48]:
check.isnull().sum().sum()

0

In [50]:
df.to_csv("cleaned_data.csv", index= False)